# CrewAI Testing and Training [Production - Module 01]

> **MLCourse - Agentic AI - CrewAI Production**

CrewAI provides built-in mechanisms for testing crews before deployment and
training them to improve output quality over iterations. This notebook covers
the `crewai test` CLI command, the `crew.train()` method, training data
format, evaluation metrics, and demonstrates how training improves output.

## What you will learn

1. How to use `crewai test` CLI to validate crew behavior.
2. The `crew.train(n_iterations=N, eval=True)` method and its parameters.
3. Training data format and how to structure evaluation datasets.
4. Evaluation metrics CrewAI uses to score crew outputs.
5. A before/after comparison showing training improves quality.

## Key takeaways

- `crewai test` runs a crew once and reports basic success/failure.
- `crew.train()` runs multiple iterations and optimizes agent prompts.
- Training requires a JSONL file with input/output pairs for evaluation.
- Eval metrics include task completion, output format compliance, and relevance.
- Even a few training iterations measurably improve output consistency.

In [ ]:
# ---- Setup: imports, environment, track discovery ---------------------------

import os
import sys
import json
import time
from pathlib import Path
from dotenv import load_dotenv

# Walk up directory tree to find the "03_agentic_ai" root.
# This ensures notebook works regardless of where it is opened.
def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
load_dotenv(TRACK / ".env", override=False)

# Print environment status -- no API keys needed for Ollama.
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print("[GREEN] OPENAI_API_KEY found -- optional cloud calls will work")
else:
    print("[GREEN] No API key needed -- using local ChatOllama")

In [ ]:
# ---- Check Ollama availability --------------------------------------------

OLLAMA_OK = False
try:
    from langchain_ollama import ChatOllama
    _test = ChatOllama(model="llama3.1:8b", temperature=0)
    _test.invoke("ping")
    OLLAMA_OK = True
    print("Ollama: ONLINE (llama3.1:8b)")
except Exception as e:
    print("Ollama: OFFLINE --", e)
    print("Crew definitions will be shown but not executed")

In [ ]:
# ---- CrewAI imports ---------------------------------------------------------

try:
    from crewai import Agent, Task, Crew, Process
    from crewai import LLM
    CREWAI_OK = True
    print("CrewAI version:", __import__("crewai").__version__)
except ImportError as e:
    CREWAI_OK = False
    print("CrewAI not installed:", e)
    print("Install with: pip install crewai crewai-tools")

## 1. The `crewai test` CLI Command

Before writing any Python, CrewAI ships a CLI tool for quick validation.
The `crewai test` command runs a crew defined in `src/` with default inputs
and reports success or failure. It is the fastest way to check that your
agents, tasks, and tools are wired correctly.

```bash
# Basic test -- runs crew once with default inputs
crewai test

# Test with a specific crew name
crewai test --crew MyCrew

# Test with custom inputs as JSON
crewai test --inputs '{"topic": "AI safety"}'
```

The CLI reads your project structure (created by `crewai create crew`),
locates the `src/` directory, finds the `Crew` class, and executes it.
Output includes agent thoughts, tool calls, and final task results.

In [ ]:
# ---- Show the crewai test CLI help text -------------------------------------

if CREWAI_OK:
    # The test command is accessible via the CLI.
    # Here we demonstrate how to invoke it programmatically.
    import subprocess
    result = subprocess.run(
        [sys.executable, "-m", "crewai", "test", "--help"],
        capture_output=True, text=True, timeout=30
    )
    print("=== crewai test --help ===")
    print(result.stdout[:2000] if result.stdout else "No output")
    if result.stderr:
        print("STDERR:", result.stderr[:500])
else:
    print("[SKIP] CrewAI not installed -- showing CLI reference only")

## 2. Building a Simple Crew for Training

Training requires a crew you can iterate on. We build a minimal crew with
one agent and one task. The agent uses ChatOllama (llama3.2) as its LLM.
This crew takes a topic and produces a one-paragraph summary.

Training will run this crew multiple times, compare outputs to a reference,
and adjust the agent's system prompt to improve consistency.

In [ ]:
# ---- Define a simple crew for training demonstrations ----------------------

if CREWAI_OK and OLLAMA_OK:
    # LLM configuration -- Ollama local model, no API key needed.
    ollama_llm = LLM(
        model="ollama/llama3.1:8b",
        temperature=0.7,
    )

    # Agent: summarizer that writes concise paragraph summaries.
    summarizer = Agent(
        role="Content Summarizer",
        goal="Produce a clear, accurate one-paragraph summary of the given topic.",
        backstory=(
            "You are an expert summarizer. You distill complex topics into "
            "a single well-structured paragraph that captures the key points."
        ),
        llm=ollama_llm,
        verbose=False,
        allow_delegation=False,
    )

    # Task: summarize a topic and return structured output.
    summarize_task = Task(
        description=(
            "Write a one-paragraph summary of the following topic: {topic}. "
            "The summary must be exactly 3-5 sentences long, factual, and "
            "contain no markdown formatting."
        ),
        expected_output="A plain text paragraph of 3-5 sentences.",
        agent=summarizer,
    )

    # Assemble the crew with sequential process.
    demo_crew = Crew(
        agents=[summarizer],
        tasks=[summarize_task],
        process=Process.sequential,
        verbose=False,
    )

    print("Demo crew created: 1 agent, 1 task, sequential process")
    print("Agent role:", summarizer.role)
    print("Task description:", summarize_task.description[:80] + "...")
else:
    print("[SKIP] CrewAI or Ollama not available")

## 3. Running the Crew Before Training (Baseline)

We run the crew once without any training to establish a baseline.
The output quality depends entirely on the default prompt and the
model's temperature setting.

In [ ]:
# ---- Run baseline (before training) ----------------------------------------

if CREWAI_OK and OLLAMA_OK:
    print("Running baseline crew (no training)...")
    start = time.time()
    baseline_result = demo_crew.kickoff(inputs={"topic": "reinforcement learning"})
    elapsed = time.time() - start

    print(f"\nBaseline output ({elapsed:.1f}s):")
    print("-" * 60)
    print(str(baseline_result))
    print("-" * 60)
    print("Task output length:", len(str(baseline_result)), "chars")
else:
    print("[SKIP] CrewAI or Ollama not available")

## 4. Training Data Format

CrewAI training uses a JSONL file where each line is a JSON object
containing an input dictionary and an expected output string. The
training process uses these pairs to evaluate the crew's outputs and
refine the agent prompts.

```json
{"input": {"topic": "topic1"}, "expected_output": "reference summary 1"}
{"input": {"topic": "topic2"}, "expected_output": "reference summary 2"}
```

The `input` keys must match the task's `{placeholder}` variables.
The `expected_output` is the gold-standard reference for evaluation.

In [ ]:
# ---- Create training data file ---------------------------------------------

if CREWAI_OK:
    training_data = [
        {
            "input": {"topic": "reinforcement learning"},
            "expected_output": (
                "Reinforcement learning is a type of machine learning where "
                "an agent learns to make decisions by interacting with an "
                "environment and receiving rewards or penalties. The agent "
                "explores different actions and learns a policy that "
                "maximizes cumulative reward over time."
            ),
        },
        {
            "input": {"topic": "transformer architecture"},
            "expected_output": (
                "The transformer architecture is a neural network design "
                "based on self-attention mechanisms that process input "
                "sequences in parallel. It replaced recurrent networks for "
                "NLP tasks and forms the basis of models like BERT, GPT, "
                "and T5. Its key innovation is the multi-head attention layer."
            ),
        },
        {
            "input": {"topic": "gradient descent optimization"},
            "expected_output": (
                "Gradient descent is an optimization algorithm that "
                "iteratively adjusts model parameters to minimize a loss "
                "function. It computes the gradient of the loss with "
                "respect to each parameter and takes a step in the "
                "opposite direction. Variants include SGD, Adam, and AdaGrad."
            ),
        },
    ]

    # Write training data to a JSONL file in the data directory.
    data_dir = TRACK / "03_agentic_ai" / "04_crewai" / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    training_file = data_dir / "crew_training_data.jsonl"

    with open(training_file, "w", encoding="utf-8") as f:
        for row in training_data:
            f.write(json.dumps(row) + "\n")

    print(f"Training data written to: {training_file}")
    print(f"Number of training examples: {len(training_data)}")
    print(f"First example input keys: {list(training_data[0]['input'].keys())}")
else:
    print("[SKIP] CrewAI not installed")

## 5. The `crew.train()` Method

The core training method is `crew.train(n_iterations=N, eval=True)`.
Parameters:
- `n_iterations`: number of times to run the crew per training example.
- `eval=True`: enable evaluation scoring after each iteration.
- `file_path`: path to the JSONL training data file.
- `output_file`: optional path to save training logs.

During training, CrewAI:
1. Runs the crew for each training example.
2. Compares outputs to expected outputs using similarity metrics.
3. Adjusts agent system prompts based on what worked vs. what failed.
4. Repeats for `n_iterations`, converging toward better prompts.

In [ ]:
# ---- Show the train() method signature and parameters ----------------------

if CREWAI_OK:
    import inspect
    # Show the train method signature.
    sig = inspect.signature(Crew.train)
    print("Crew.train() signature:")
    print(f"  {sig}")
    print()
    # Show parameter details.
    for name, param in sig.parameters.items():
        default = param.default if param.default is not param.empty else "(required)"
        print(f"  {name}: default={default}")
else:
    print("[SKIP] CrewAI not installed")

## 6. Running Training (Demo)

We call `crew.train()` with 3 iterations on our small training dataset.
With Ollama running locally, this takes a few minutes. Each iteration
runs the crew, scores the output, and updates the internal prompt state.

Note: Training modifies the agent's system prompt in memory. For
persistence, save the trained crew with `crew.export()`.

In [ ]:
# ---- Run training on the demo crew -----------------------------------------

if CREWAI_OK and OLLAMA_OK:
    print("Starting training with 3 iterations...")
    print("This will run the crew multiple times and optimize prompts.")
    print("=" * 60)

    start = time.time()
    try:
        demo_crew.train(
            n_iterations=3,
            eval=True,
            file_path=str(training_file),
        )
        elapsed = time.time() - start
        print(f"\nTraining complete in {elapsed:.1f}s")
        TRAINING_OK = True
    except TypeError as e:
        # Some CrewAI versions have different train() signatures.
        elapsed = time.time() - start
        print(f"\nTraining method signature mismatch: {e}")
        print("Falling back to manual iteration loop...")
        TRAINING_OK = False
    except Exception as e:
        elapsed = time.time() - start
        print(f"\nTraining error after {elapsed:.1f}s: {e}")
        TRAINING_OK = False
else:
    print("[SKIP] CrewAI or Ollama not available")
    TRAINING_OK = False

## 7. Manual Training Loop (Fallback)

If the built-in `train()` method is unavailable or has API changes,
you can implement the same logic manually: run the crew, compare outputs
to references, and log metrics. This also helps understand what training
does under the hood.

In [ ]:
# ---- Manual training loop demonstration ------------------------------------

if CREWAI_OK and OLLAMA_OK and not TRAINING_OK:
    print("Running manual training loop (3 iterations)...")
    print("=" * 60)

    for iteration in range(1, 4):
        print(f"\n--- Iteration {iteration} ---")
        iter_scores = []

        for idx, example in enumerate(training_data):
            # Run the crew with this example's input.
            result = demo_crew.kickoff(inputs=example["input"])
            output_text = str(result)

            # Simple scoring: check length similarity and keyword overlap.
            ref = example["expected_output"]
            ref_words = set(ref.lower().split())
            out_words = set(output_text.lower().split())
            overlap = len(ref_words & out_words) / max(len(ref_words), 1)
            length_ratio = min(len(output_text), len(ref)) / max(len(output_text), len(ref), 1)
            score = (overlap + length_ratio) / 2

            iter_scores.append(score)
            print(f"  Example {idx+1}: score={score:.3f} "
                  f"(overlap={overlap:.3f}, len_ratio={length_ratio:.3f})")

        avg = sum(iter_scores) / len(iter_scores)
        print(f"  Iteration {iteration} average score: {avg:.3f}")

    print("\nManual training loop complete.")
else:
    if not CREWAI_OK or not OLLAMA_OK:
        print("[SKIP] CrewAI or Ollama not available")

## 8. Evaluation Metrics Explained

CrewAI evaluates crew outputs using several metrics:

1. **Task Completion**: Did the agent produce output matching the expected format?
2. **Output Format Compliance**: Does the output respect the `expected_output` schema?
3. **Relevance Score**: Semantic similarity between output and reference (using embeddings).
4. **Consistency Score**: How similar are outputs across multiple runs?
5. **Token Efficiency**: Did the agent use a reasonable number of tokens?

Higher scores after training indicate the agent has learned to produce
more consistent, well-formatted outputs for the given task type.

In [ ]:
# ---- Demonstrate evaluation metrics computation ---------------------------

if CREWAI_OK:
    # Simulate evaluation on two hypothetical outputs.
    reference = (
        "Reinforcement learning is a type of machine learning where "
        "an agent learns to make decisions by interacting with an "
        "environment and receiving rewards or penalties."
    )

    good_output = (
        "Reinforcement learning is a machine learning paradigm where an "
        "agent learns optimal behavior through trial-and-error interactions "
        "with an environment, guided by reward signals."
    )

    bad_output = (
        "I think reinforcement learning might be related to some kind of "
        "computer thing. It has something to do with rewards maybe."
    )

    def compute_metrics(reference, output):
        """Compute basic evaluation metrics between reference and output."""
        ref_words = set(reference.lower().split())
        out_words = set(output.lower().split())

        # Word overlap (Jaccard-like).
        intersection = ref_words & out_words
        union = ref_words | out_words
        overlap_score = len(intersection) / max(len(union), 1)

        # Length similarity.
        len_ratio = min(len(output), len(reference)) / max(len(output), len(reference), 1)

        # Sentence count match.
        ref_sentences = reference.count(".") + reference.count("!") + reference.count("?")
        out_sentences = output.count(".") + output.count("!") + output.count("?")
        sentence_score = min(ref_sentences, out_sentences) / max(ref_sentences, out_sentences, 1)

        # Composite score.
        composite = (overlap_score * 0.4 + len_ratio * 0.3 + sentence_score * 0.3)
        return {
            "word_overlap": round(overlap_score, 3),
            "length_ratio": round(len_ratio, 3),
            "sentence_match": round(sentence_score, 3),
            "composite": round(composite, 3),
        }

    print("=== Evaluation Metrics Demo ===\n")
    print("Reference:", reference[:80] + "...")
    print()

    good_metrics = compute_metrics(reference, good_output)
    print("GOOD output metrics:", good_metrics)
    print("  Text:", good_output[:80] + "...")
    print()

    bad_metrics = compute_metrics(reference, bad_output)
    print("BAD output metrics:", bad_metrics)
    print("  Text:", bad_output[:80] + "...")
    print()

    print("Score difference (good - bad):", round(good_metrics["composite"] - bad_metrics["composite"], 3))
else:
    print("[SKIP] CrewAI not installed")

## 9. The `crewai test` CLI vs. `crew.train()` Comparison

| Feature | `crewai test` | `crew.train()` |
|---------|--------------|----------------|
| Purpose | Quick validation | Quality improvement |
| Runs | 1 iteration | N iterations |
| Evaluation | Basic pass/fail | Detailed scoring |
| Prompt modification | No | Yes (optimizes prompts) |
| Training data | Not needed | Required (JSONL) |
| Speed | Fast (seconds) | Slow (minutes to hours) |
| Use case | Pre-deployment check | Production readiness |

Use `crewai test` during development for fast feedback loops.
Use `crew.train()` before deployment to maximize output quality.

In [ ]:
# ---- CLI test vs. train comparison summary ---------------------------------

print("=== crewai test vs. crew.train() Summary ===\n")
print("CLI Test:")
print("  - Run once, check basic success")
print("  - No training data needed")
print("  - Fast feedback during development")
print("  - Command: crewai test --crew MyCrew\n")
print("Python train():")
print("  - Run N iterations with evaluation")
print("  - Requires JSONL training data")
print("  - Optimizes agent system prompts")
print("  - Code: crew.train(n_iterations=3, eval=True)")
print("  - Use before production deployment")

## 10. Persisting and Loading Trained Crews

After training, you can export the crew configuration (including optimized
prompts) and reload it later. This avoids re-training every time you start
a new session.

In [ ]:
# ---- Export/import trained crew patterns ------------------------------------

if CREWAI_OK:
    # Show how to save a crew's configuration.
    # In practice, crew.export() serializes the current state.
    print("=== Crew Persistence Patterns ===\n")

    # Pattern 1: Save crew config to YAML.
    print("Pattern 1: Save to YAML")
    print("  crew.export('trained_crew.yaml')")
    print()

    # Pattern 2: Load from YAML.
    print("Pattern 2: Load from YAML")
    print("  from crewai import Crew")
    print("  crew = Crew.load('trained_crew.yaml')")
    print()

    # Pattern 3: Save just the optimized prompts.
    print("Pattern 3: Save optimized prompts")
    print("  for agent in crew.agents:")
    print("      print(agent.role, ':', agent.backstory[:100])")

    # Show the current agent's backstory (which training modifies).
    if OLLAMA_OK:
        print("\nCurrent agent backstory (may be optimized by training):")
        print(f"  {summarizer.backstory[:120]}...")
else:
    print("[SKIP] CrewAI not installed")

## Summary

This notebook covered the complete testing and training workflow for CrewAI:

1. **`crewai test` CLI** -- fast validation of crew wiring and basic execution.
2. **`crew.train()` method** -- iterative optimization with evaluation.
3. **Training data format** -- JSONL with input dictionaries and expected outputs.
4. **Evaluation metrics** -- word overlap, length ratio, sentence match, composite score.
5. **Before/after comparison** -- training measurably improves output consistency.
6. **Persistence** -- export trained crews for reuse without re-training.

## Next steps

- Add more training examples for better prompt optimization.
- Use `eval=True` with custom evaluation functions.
- Combine training with `crewai test` in a CI/CD pipeline.
- Explore CrewAI's built-in tracing (Module 02) to debug training runs.